In [1]:
!hostname

node002


In [2]:
pwd

'/work/cxiao'

In [3]:
import hyperalignment
print(hyperalignment.__file__)

/work/cxiao/hyperalignment/src/hyperalignment/__init__.py


In [3]:
!pip install --upgrade git+https://github.com/feilong/neuroboros.git

  Cloning https://github.com/feilong/neuroboros.git to /local/cxiao/pip-req-build-1evljfla
  Running command git clone --filter=blob:none --quiet https://github.com/feilong/neuroboros.git /local/cxiao/pip-req-build-1evljfla
  Resolved https://github.com/feilong/neuroboros.git to commit 4747dde0f5e7c55f15e466a24099f3c1c4a64494
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
import os
import pandas as pd
import re
import neuroboros as nb
import numpy as np
from scipy.stats import zscore
from hyperalignment import (initialize_sparse_matrix, searchlight_procrustes,
                            searchlight_weights)
from scipy.spatial.distance import cdist
from hyperalignment.local_template import compute_template
from scipy.sparse import block_diag,csr_matrix
import hyperalignment as ha
from hyperalignment.searchlight import searchlight_procrustes

In [3]:
import neuroboros as nb
import inspect
print(inspect.signature(nb.plot2d.brain_plot))

(values, cmap=None, vmax=None, vmin=None, alpha=None, space=None, mask=None, surf_type='inflated', nn=True, return_scale=False, background_color=[1.0, 1.0, 1.0, 0.0], gyri_color=[0.8, 0.8, 0.8, 1.0], sulci_color=[0.6, 0.6, 0.6, 1.0], colorbar=True, output=None, scale=None, title=None, title_size=70, fn=None, parc=None, parc_kwargs=None, **kwargs)


In [3]:
help(searchlight_procrustes)

Help on function searchlight_procrustes in module hyperalignment.searchlight:

searchlight_procrustes(
    X,
    Y,
    sls,
    sls_Y=None,
    mat0=None,
    reflection=True,
    scaling=False,
    weights=None
)
    Searchlight hyperalignment using orthogonal Procrustes.

    Parameters
    ----------
    X : ndarray of shape (n_samples, n_features_X)
        The first data matrix, to be aligned to ``Y``.
    Y : ndarray of shape (n_samples, n_features_Y)
        The second data matrix, the target for alignment.
    sls : list of ndarrays
        Each ndarray contains the indices of the vertices in the searchlight
        for the first data matrix.
    sls_Y : list of ndarrays or None, default=None
        Searchlight indices for the second data matrix, if different from
        ``sls``.
    mat0 : sparse matrix or None, default=None
        The sparse matrix to initialize the transformation, because changing
        sparse matrix sparsity is very slow. If None, initialize using a 

In [6]:
class NEP(nb.datasets.Dataset):
    def __init__(
        self,
        name="NEP",
        space=["onavg-ico32"],
        resample=["1step_pial_overlap"],
        prep="default",
        fp_version="25.1.4",
        dl_source=None,
        root_dir="/work/cxiao/NEP/nb-data",
    ):
        super().__init__(
            name,
            dl_source=dl_source,
            root_dir=root_dir,
            space=space,
            resample=resample,
            prep=prep,
            fp_version=fp_version,
        )
        self.subjects = self.subject_sets["all"]

    def rename_func(self, sid, task, run, suffix=".npy"):
        return f"sub-{sid}_task-{task}_run-{run:02d}{suffix}"

    def load_confounds(self, sid, task, run, fp_version):
        """
        Overrides the default behavior to match your specific filename:
        task-sen_run-01_desc-confounds_timeseries.npy
        """
        # 1. Construct the filename with zero-padding (:02d)
        filename = f"task-{task}_run-{run:02d}_desc-confounds_timeseries.npy"
        
        # 2. Build the full path (Fixing self.root to self.root_dir)
        confound_path = os.path.join(self.root_dir, fp_version, 'confounds', filename)
        
        if not os.path.exists(confound_path):
            # Fallback check for 'sub-' prefix
            alt_path = os.path.join(self.root_dir, fp_version, 'confounds', f"sub-{sid}_{filename}")
            if os.path.exists(alt_path):
                confound_path = alt_path
            else:
                raise RuntimeError(f"Confounds file not found at:\n{confound_path}\nor\n{alt_path}")
        
        # 3. Load and return as a list
        confounds = np.load(confound_path)
        return [confounds]

In [7]:
if __name__ == "__main__":
    radius = 20
    sls = nb.sls("lr", radius)
    mat0 = nb.record(
        f"/work/cxiao/NEP/HA/mat0_{radius}mm.npz",
        initialize_sparse_matrix,
        return_results=True,
    )(sls)
    print(mat0.shape)
    tpl = [
        np.load(f"/work/cxiao/sparta-gd_nll8.0_50000/conn_zscored_{lr}h.npy")
        for lr in "lr"
    ]
    tpl = np.concatenate(tpl, axis=1)
    tpl = tpl @ nb.mapping("lr", "onavg-ico64", "onavg-ico32")
    tpl = tpl[nb.mask("lr", "onavg-ico8")]
    tpl = tpl[:, nb.mask("lr", "onavg-ico32")]
    print(tpl.shape)
    

2026-03-26 14:32:11.725048 `finish_fn` exists: /work/cxiao/NEP/HA/mat0_20mm.npz.finish
(19341, 19341)
(1210, 19341)


In [9]:
dset = NEP()
sids = dset.subjects


csv_path = "/work/cxiao/NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat/highest_correlated_runs_ts_and_beta.csv"
df_best = pd.read_csv(csv_path)

# All possible runs exactly as they appear in nb-data
all_possible_runs = {"wd-01", "wd-02", "sen-01", "sen-02"}

def standardize_run_str(run_str):
    """Converts varying formats like 'wd_01', 'wd1', or 'wd-1' into standard 'wd-01'"""
    match = re.match(r"([a-zA-Z]+)[_-]?0*(\d+)", str(run_str).strip())
    if match:
        task_name = match.group(1) # 'wd' or 'sen'
        run_num = match.group(2)   # '1' or '2'
        return f"{task_name}-0{run_num}" 
    return None

# Dictionary to collect our assignments for verification
run_assignments = {}

print("Parsing CSV and assigning training runs...")

for sid in sids:
    subj_row = df_best[df_best['Subject'].astype(str) == str(sid)]
    
    if subj_row.empty:
        print(f"Warning: Subject {sid} not found in CSV. Skipping.")
        continue
        
    # Extract raw strings from CSV
    raw_best_A = subj_row['TS_Best_Run_A'].values[0]
    raw_best_B = subj_row['TS_Best_Run_B'].values[0]
    
    # Standardize to match nb-data
    best_runs = {
        standardize_run_str(raw_best_A), 
        standardize_run_str(raw_best_B)
    }
    

    training_runs = list(all_possible_runs - best_runs)
    training_runs.sort() # Keep output clean and predictable
    
    # Save the logic to our dictionary
    run_assignments[sid] = {
        "CSV_Raw_A": raw_best_A,
        "CSV_Raw_B": raw_best_B,
        "Parsed_Best": ", ".join(best_runs),
        "Training_Runs_To_Load": ", ".join(training_runs)
    }


df_verify = pd.DataFrame.from_dict(run_assignments, orient='index')
df_verify.index.name = 'Subject'

print("\n--- ASSIGNMENT VERIFICATION TABLE ---")
print(df_verify.head(10)) # Print the first 10 rows to inspect

#df_verify.to_csv("/work/cxiao/NEP/HA/training_run_assignments_log.csv")

Parsing CSV and assigning training runs...

--- ASSIGNMENT VERIFICATION TABLE ---
        CSV_Raw_A CSV_Raw_B    Parsed_Best Training_Runs_To_Load
Subject                                                         
3102        wd_01     wd_02   wd-02, wd-01        sen-01, sen-02
3103        wd_01    sen_01  wd-01, sen-01         sen-02, wd-02
3104        wd_01    sen_01  wd-01, sen-01         sen-02, wd-02
3105        wd_01    sen_01  wd-01, sen-01         sen-02, wd-02
3106        wd_01    sen_01  wd-01, sen-01         sen-02, wd-02
3107        wd_02    sen_01  wd-02, sen-01         sen-02, wd-01
3108        wd_01    sen_01  wd-01, sen-01         sen-02, wd-02
3109        wd_02    sen_02  wd-02, sen-02         sen-01, wd-01
3110        wd_01     wd_02   wd-02, wd-01        sen-01, sen-02
3111        wd_01    sen_01  wd-01, sen-01         sen-02, wd-02


In [10]:
#compute xmf

base_out_dir = "/work/cxiao/NEP/HA"
os.makedirs(base_out_dir, exist_ok=True)

task = "tb" 
w = searchlight_weights(sls, radius=radius)


total_sids = len(sids)

for i, sid in enumerate(sids, start=1):
    print(f"\n[{i}/{total_sids}] ==========================================")
    print(f"[{i}/{total_sids}] Starting Subject: {sid}")
    
    # Construct the output filename
    out_dir = os.path.join(base_out_dir, "xfms", str(sid))
    out_fn = os.path.join(out_dir, f"{task}_{radius}mm_to-tpl.npz")
    
    if os.path.exists(out_fn):
        print(f"[{i}/{total_sids}] -> Skipping {sid}, transform already exists.")
        continue

    # 1. Look up which runs to use for this specific subject
    if sid not in run_assignments:
        print(f"[{i}/{total_sids}] -> Skipping {sid}, no run assignments found.")
        continue
        
    training_run_list = run_assignments[sid]["Training_Runs_To_Load"].split(", ")
    
    # 2. Load and concatenate the two training runs
    dm_list = []
    print(f"[{i}/{total_sids}] -> Loading runs: {training_run_list}")
    
    for run_str in training_run_list:
        t_name, r_str = run_str.split('-')
        r_num = int(r_str)
        
        run_data = dset.get_data(sid, t_name, r_num, "lr")
        dm_list.append(run_data)
    
    # Stack the runs vertically (concatenate in time)
    dm = np.concatenate(dm_list, axis=0)
    print(f"[{i}/{total_sids}] -> Data loaded. Full dm shape: {dm.shape}")

    # 3. Compute Connectivity and Hyperalignment
    print(f"[{i}/{total_sids}] -> Downsampling to targets...")
    targets = dm @ nb.mapping("lr", "onavg-ico32", "onavg-ico8", mask=True)
    
    print(f"[{i}/{total_sids}] -> Computing connectivity matrix (cdist)...")
    conn = 1 - cdist(targets.T, dm.T, "correlation")
    conn = np.nan_to_num(zscore(conn, axis=0))
    print(f"[{i}/{total_sids}] -> Conn shape: {conn.shape}, Tpl shape: {tpl.shape}")


    print(f"[{i}/{total_sids}] -> Running Searchlight Procrustes (this may take a few minutes)...")
    xfm = searchlight_procrustes(conn, tpl, sls, weights=w, mat0=mat0)
   

    os.makedirs(out_dir, exist_ok=True)
    nb.save(out_fn, xfm)
    print(f"[{i}/{total_sids}] -> SUCCESS! Saved xfm to {out_fn}")


[1/30] ==========================================
[1/30] Starting Subject: 3102
[1/30] -> Skipping 3102, transform already exists.

[2/30] ==========================================
[2/30] Starting Subject: 3103
[2/30] -> Skipping 3103, transform already exists.

[3/30] ==========================================
[3/30] Starting Subject: 3104
[3/30] -> Skipping 3104, transform already exists.

[4/30] ==========================================
[4/30] Starting Subject: 3105
[4/30] -> Skipping 3105, transform already exists.

[5/30] ==========================================
[5/30] Starting Subject: 3106
[5/30] -> Skipping 3106, transform already exists.

[6/30] ==========================================
[6/30] Starting Subject: 3107
[6/30] -> Skipping 3107, transform already exists.

[7/30] ==========================================
[7/30] Starting Subject: 3108
[7/30] -> Skipping 3108, transform already exists.

[8/30] ==========================================
[8/30] Starting Subject: 

In [11]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp  

def load_and_align_data(base_dir, dset):
    csv_path = os.path.join(base_dir, 'training_run_assignments_log.csv')
    log_df = pd.read_csv(csv_path)
    log_df['Subject'] = log_df['Subject'].astype(str) 

    # Create the folder to save aligned data
    save_dir = os.path.join(base_dir, 'aligned')
    os.makedirs(save_dir, exist_ok=True) 

    aligned_datasets = {}

    for _, row in log_df.iterrows():
        sub_id = row['Subject']
        
        if sub_id not in dset.subjects:
            print(f"Skipping Subject {sub_id}: Not found in dataset.")
            continue
            
        # Split the comma-separated runs into a list (e.g., ['wd-01', 'sen-01'])
        raw_runs_string = str(row['Parsed_Best'])
        runs_to_process = [r.strip() for r in raw_runs_string.split(',')]
        
        # Load the subject's transformation matrix ONCE per subject
        xfm_path = os.path.join(base_dir, 'xfms', sub_id, 'tb_20mm_to-tpl.npz') 
        
        try:
            # --- THE SPARSE MATRIX FIX ---
            # Rebuild the 2D matrix from the zip file components
            xfm_sparse = sp.load_npz(xfm_path)
            xfm = xfm_sparse.toarray() 
        except FileNotFoundError:
            print(f"  ERROR: Could not find xfm file for {sub_id}")
            continue
        except Exception as e:
            print(f"  ERROR loading xfm for {sub_id}: {e}")
            continue

        # Loop through the individual test runs for this subject
        for run_id in runs_to_process:
            print(f"Processing Subject: {sub_id} | Aligning Run: {run_id}")
            
            try:
                # Parse the run string (e.g., "wd-01" -> "wd", 1)
                t_name, r_str = run_id.split('-')
                r_num = int(r_str)
                
                # Fetch the brain data
                data = dset.get_data(sub_id, t_name, r_num, "lr")
                
                # Shape check: ensure it is (Timepoints, Voxels)
                if data.shape[1] != xfm.shape[0]:
                    data = data.T
                    
                # --- THE ALIGNMENT ---
                # This works perfectly whether data has 440 or 454 timepoints!
                aligned_data = data @ xfm
                
                # Store in memory
                dict_key = f"{sub_id}_{run_id}"
                aligned_datasets[dict_key] = aligned_data
                
                # Save to disk
                save_filename = f"{sub_id}_{run_id}_aligned.npy"
                save_path = os.path.join(save_dir, save_filename)
                np.save(save_path, aligned_data)
                
                print(f"  Success! Shape: {aligned_data.shape} | Saved: {save_filename}")

            except Exception as e:
                print(f"  ERROR for {sub_id} run {run_id}: {e}")

    return aligned_datasets

if __name__ == "__main__":
    # Assuming dset is already defined in your environment
    HA_BASE_DIR = '/work/cxiao/NEP/HA'
    results_dict = load_and_align_data(HA_BASE_DIR, dset)

Processing Subject: 3102 | Aligning Run: wd-01
  Success! Shape: (440, 19341) | Saved: 3102_wd-01_aligned.npy
Processing Subject: 3102 | Aligning Run: wd-02
  Success! Shape: (440, 19341) | Saved: 3102_wd-02_aligned.npy
Processing Subject: 3103 | Aligning Run: sen-01
  Success! Shape: (454, 19341) | Saved: 3103_sen-01_aligned.npy
Processing Subject: 3103 | Aligning Run: wd-01
  Success! Shape: (440, 19341) | Saved: 3103_wd-01_aligned.npy
Processing Subject: 3104 | Aligning Run: sen-01
  Success! Shape: (454, 19341) | Saved: 3104_sen-01_aligned.npy
Processing Subject: 3104 | Aligning Run: wd-01
  Success! Shape: (440, 19341) | Saved: 3104_wd-01_aligned.npy
Processing Subject: 3105 | Aligning Run: sen-01
  Success! Shape: (454, 19341) | Saved: 3105_sen-01_aligned.npy
Processing Subject: 3105 | Aligning Run: wd-01
  Success! Shape: (440, 19341) | Saved: 3105_wd-01_aligned.npy
Processing Subject: 3106 | Aligning Run: sen-01
  Success! Shape: (454, 19341) | Saved: 3106_sen-01_aligned.npy
Pr